In [1]:
import tensorflow as tf
from tensorflow import keras
#import matplotlib.pyplot as plt
import numpy as np

In [2]:
dataset = tf.keras.utils.image_dataset_from_directory(
    "E:/Paddy_detect/archive",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

Found 10407 files belonging to 10 classes.
Using 8326 files for training.


In [3]:
class_names=dataset.class_names
print(class_names)

['bacterial_leaf_blight', 'bacterial_leaf_streak', 'bacterial_panicle_blight', 'blast', 'brown_spot', 'dead_heart', 'downy_mildew', 'hispa', 'normal', 'tungro']


In [4]:
data_dir = "E:/Paddy_detect/archive"

In [5]:
normalization_layer=keras.layers.Rescaling(2./255)

In [6]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir ,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

Found 10407 files belonging to 10 classes.
Using 8326 files for training.
Found 10407 files belonging to 10 classes.
Using 2081 files for validation.


In [7]:
val_ds

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [8]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [9]:
model = keras.Sequential([
    normalization_layer,

    keras.layers.Conv2D(32, 3, activation='relu'),
    keras.layers.MaxPooling2D(),

    keras.layers.Conv2D(64, 3, activation='relu'),
    keras.layers.MaxPooling2D(),

    keras.layers.Conv2D(128, 3, activation='relu'),
    keras.layers.MaxPooling2D(),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(len(class_names), activation='softmax')
])

In [10]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 205s 761ms/step - accuracy: 0.3356 - loss: 1.9614 - val_accuracy: 0.4551 - val_loss: 1.5810
Epoch 2/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 195s 746ms/step - accuracy: 0.5747 - loss: 1.2992 - val_accuracy: 0.6401 - val_loss: 1.1564
Epoch 3/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 203s 779ms/step - accuracy: 0.7430 - loss: 0.8067 - val_accuracy: 0.6386 - val_loss: 1.1304
Epoch 4/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 224s 859ms/step - accuracy: 0.8503 - loss: 0.4723 - val_accuracy: 0.7150 - val_loss: 1.1390
Epoch 5/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 236s 757ms/step - accuracy: 0.9141 - loss: 0.2778 - val_accuracy: 0.7593 - val_loss: 1.0587
Epoch 6/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 171s 654ms/step - accuracy: 0.9508 - loss: 0.1593 - val_accuracy: 0.7852 - val_loss: 1.2380
Epoch 7/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 230s 881ms/step - accuracy: 0.9643 - loss: 0.1169 - val_accuracy: 0.7420 - val_loss: 1.4219
Epoch 8/10
261/261 ━━━━━━━━━━━━━━━━━━━━ 265s 1s/step - accuracy: 0.9670 - lo

In [12]:
img = tf.keras.utils.load_img(
    "archive/tungro/100461.jpg",
    target_size=(224, 224)
)

In [13]:
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

In [14]:
prediction=model.predict(img_array)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step


In [15]:
score = tf.nn.softmax(prediction[0])

print("Disease:",
      class_names[np.argmax(score)])

print("Confidence:",
      100 * np.max(score), "%")

Disease: tungro
Confidence: 23.196926 %


In [16]:
model.save("paddy_disease_model1.h5")